# Project 2

Welcome to the second NLP1 project. The goal of the second tutorial was to get you familiar with using pre-trained embeddings and moving towards a compositional encoder for text classification.

Given the skills you have developed in the last week, you will now be applying them within the dataset and task you have chosen during Project 1. You will be applying hands-on the experiments you have already explored in the Tutorial 2, and taking a deeper dive: extending your experiments, trying different set ups, possibly considering different evaluation techniques and errors analyses, and tackling your own research questions. 


The week after the deadline, your TA will perform a short oral assessment of the skills and knowledge you acquired (make sure to check the official schedule for this), probing you to answer questions about what you learnt, about what you did and about why you did the things you did (as opposed to some reasonable alternatives) as well as pros and cons of your method, and so on. 

# Get to work!

1. Find a word embedding model that is appropriate for your dataset. Think about the language, the kind of text you are working with. Find evidence that your choice is a good one: 
    * Can you tokenize your texts in a way coherent with the definition of a token in the embedding model?
    * Does the model's vocabulary give enough coverage to the words in your texts? etc.
2. Explore what your chosen word embedding model captures. Some things you might consider: 
    * Is it good at representing interesting similarities (re to your choice of task)? J
    * Have you found task-specific evidence that it's better than some other reasonable alternative?
    * Do you know about limitations of this model?
    * Any limitations that you find particularly troubling for your task? 
3. Experiment with the embed-and-pool approach.  
    * What design choices have meaningful impact in classification performance (eg, embedding size, architecture design, optimisation parameters), if any?
    * Is this better than what you could achieve last week? In other words, did pretrained embeddings add to your design?
4. Experiment with a recurrent encoder (GRU or LSTM). 
    * What design choices have meaningful impact in classification performance, if any?
    * Did you investigate the impact of data size, embedding parameters (trained or not, fixed or trainable, etc)?
    * Did you struggle with over-fitting?
    * How far can this approach go? Are you able to improve meaningfully on embed-and-pool?
5. We make decisions under uncertainty all the time, are you gathering enough information to substantiate your choices? Various things can lead to small differences in results, and these differences are not always consistent. Here are some things to think about:
    * How did you structure you hyperparameter search?
    * Are you keeping an eye on experimental variance?
6. Experiment with error analysis and try to discover what characteristics of input texts are predictive of performance differences across major design choices. 
    * What analysis features did you come up with?
    * Which ones did you manage to test and how did that go?
    * What would you prioritise in the future in order to improve your systems?

Be mindful that you will be using your experiments to write up your final blog, so make sure your code is clean and neat!

If you are unsure about anything, ask your TA! They are always happy to help you! Happy coding! :)

# Tips

## Pretrained embedding models

We will post a list with some options for you. 

## Features for analysis

Features that you can interpret as an aspect of _complexity_ in light of your task's goals.

- length
- presence of rare/common words, including unknown words
- syntactic features (like parts-of-speech, syntactic complexity, syntactic voice, presence of negation, etc.)
- semantic features (eg, the average similarity of the text to training documents or some curated set of training documents, presence of words in certain semantic classes, etc.)
- task-specific features (eg, 
- model-specific features (eg, predicted label, probability assigned to predicted label, number of parameters or layers or some other model descriptor)

In [1]:
%load_ext autoreload
%autoreload 2    

In [2]:
import arviz as az
import sys
from pathlib import Path
root = Path.cwd().parent
sys.path.insert(0, str(root))
from src import *

import part1  # we will reuse some things from last tutorial
import part2

In [3]:
# save models and outputs there to keep the directory tree organised
results_folder = Path("./results")
results_folder.mkdir(parents=True, exist_ok=True) 
results_folder

WindowsPath('results')

In [4]:
from part1 import *

train, validation, test = load_sst2()
print(CLASSES)
print("train:", len(train))
print("validation:", len(validation))
print("test:", len(test))
print(train[2])

0: negative, 1: positive
train: 67349
validation: 872
test: 1821
('that loves its characters and communicates something rather beautiful about human nature ', 1)


<h1>Choosing embedding model<h1>

In [5]:
%%time
# depends on internet connection (but say 5 minutes or less)
embtable_version, embtable_dims, embtable_metadata = download_pretrained('fasttext') # in case stanford.edu is down, use fasttext (but glove is preferred)
embtable_version, embtable_dims, embtable_metadata
trained_emb_model = part2.clslib.PretrainedEmbeddings(embtable_version, embtable_dims, skip_first_line=embtable_metadata)

CPU times: total: 1min 7s
Wall time: 1min 8s


In [6]:
from collections import Counter
from part2.clslib import *
import part1.experiment as exp1
import part2.experiment as exp2

tokenizer = BasicTokenizer(
    lowercase=True,
    remove_punct=False,
    remove_stopwords=False
)

all_tokens = [
    token
    for sentence, label in train
    for token in tokenizer.tokens(sentence)
]

word_counts = Counter(all_tokens)

print("Total tokens:", len(all_tokens))
print("Unique tokens:", len(word_counts))

Total tokens: 634100
Unique tokens: 14801


In [7]:
vocab = trained_emb_model.vocab

covered_types = sum(
    word in vocab
    for word in word_counts
)

type_coverage = covered_types / len(word_counts)

covered_tokens = sum(
    count
    for word, count in word_counts.items()
    if word in vocab
)

token_coverage = covered_tokens / sum(word_counts.values())

print(f"Type coverage: {type_coverage:.2%}")
print(f"Token coverage: {token_coverage:.2%}")

Type coverage: 95.68%
Token coverage: 98.62%


# Glove

>1. 90.66% of all unique tokens in dataset occurs in Glove
>2. 98.30% of all tokens in dataset occurs in Glove

# Fasttext

>1. 95.61% of all unique tokens in dataset occurs in Glove
>2. 98.60% of all tokens in dataset occurs in Glove

# ConceptNet
>1. 86.07% of all unique tokens in dataset occurs in Glove
>2. 88.08% of all tokens in dataset occurs in Glove

# Explore word embedding model

In [8]:
import nltk
from nltk.corpus import opinion_lexicon

In [9]:
positive_words = opinion_lexicon.positive()
negative_words = opinion_lexicon.negative()

positive_candidates = set(positive_words) & set(word_counts) & set(trained_emb_model.vocab)
negative_candidates = set(negative_words) & set(word_counts) & set(trained_emb_model.vocab)

top_positive = sorted(
    positive_candidates,
    key=lambda w: word_counts[w],
    reverse=True
)

top_negative = sorted(
    negative_candidates,
    key=lambda w: word_counts[w],
    reverse=True
)

print("Positive:", len(top_positive), top_positive[:10])
print("Negative:", len(top_negative), top_negative[:10])

Positive: 983 ['like', 'good', 'best', 'love', 'work', 'enough', 'humor', 'well', 'great', 'better']
Negative: 1621 ['funny', 'bad', 'plot', 'hard', 'dull', 'worst', 'dark', 'silly', 'boring', 'dumb']


In [10]:
print(type(trained_emb_model.vocab))
print(type(trained_emb_model.vectors))
print(trained_emb_model.vectors.shape)

<class 'part2.clslib.Vocabulary'>
<class 'numpy.ndarray'>
(830463, 300)


In [11]:
positive_positive = average_similarity(combinations(top_positive, 2), trained_emb_model)
negative_negative = average_similarity(combinations(top_negative, 2), trained_emb_model)
positive_negative = average_similarity(product(top_positive, top_negative), trained_emb_model)

print("Positive-Positive:", positive_positive)
print("Negative-Negative:", negative_negative)
print("Positive-Negative:", positive_negative)

Positive-Positive: 0.33206913
Negative-Negative: 0.32115144
Positive-Negative: 0.29655498


# Glove
>1. Positive-Positive: 0.32262674
>2. Negative-Negative: 0.2682382
>3. Positive-Negative: 0.23892917

# Fasttext
>1. Positive-Positive: 0.33206916
>2. Negative-Negative: 0.32115144
>3. Positive-Negative: 0.29655498

# ConceptNet
>1. Positive-Positive: 0.08545099
>2. Negative-Negative: 0.07145761
>3. Positive-Negative: 0.03152869

1. **Sentiment similarities:** GloVe captures some sentiment information, as words with the same sentiment have higher average cosine similarity than words with opposite sentiment.

2. **Comparison:** GloVe shows the largest sentiment separation (0.0565), compared to ConceptNet (0.0470) and fastText (0.0301).

3. **Limitations:** GloVe uses static embeddings, words have the same representation regardless of context. It also cannot directly represent OOV words.

4. **Task-specific limitations:** Negation and context-dependent sentiment are problematic for sentiment classification. These limitations also apply to fastText and ConceptNet.

# Embed and pool

In [ ]:
# from nltk.tokenize import TreebankWordTokenizer
# import smlp2
# from src import *

# word_tokenizer = TreebankWordTokenizer()

# selected_encoders = ["mean-embedding"]
# model_name = "mlp2"

# encoder = MeanPoolingEncoder(
#     tokenizer=word_tokenizer,
#     embedding_model=trained_emb_model
# )


In [12]:
from part2.clslib import *
import part1.experiment as exp1
import part2.experiment as exp2
tokenizer = BasicTokenizer(
    lowercase=True,
    remove_punct=False,
    remove_stopwords=False,
    vocab=trained_emb_model.vocab
)

In [47]:
from pathlib import Path
import torch
import numpy as np
import json

method = "fasttext_embed_and_pool"

enc_kwargs = {
    "freeze_embeddings": True
}

regressor_kwargs = {
    "hidden_size1": 256,
    "hidden_size2": 128,
    "dropout": 0.3
}

opt_kwargs = {
    "learning_rate": 0.001,
    "weight_decay": 1e-4,
    "batch_size": 64,
    "num_epochs": 10
}

save_dir = Path(f"results/embed_and_pool/cross_validation/{method}")
save_dir.mkdir(parents=True, exist_ok=True)

train, validation, test = load_sst2()

data_splits = exp1.cross_validation_split(
    pairs=train,
    k=5,
    seed=0
)

results_embed_pool = exp2.cross_validation(
    outcome_space=CLASSES,
    data_splits=data_splits,
    tokenizer=tokenizer,
    emb_model=trained_emb_model,
    enc_type="embed-and-pool",
    enc_kwargs=enc_kwargs,
    regressor_kwargs=regressor_kwargs,
    opt_kwargs=opt_kwargs,
    device="cuda" if torch.cuda.is_available() else "cpu",
    save_to=str(save_dir)
)

accuracies = list(results_embed_pool["accuracies"].values())

results = {
    "accuracy_per_fold": accuracies,
    "mean_accuracy": float(np.mean(accuracies)),
    "std_accuracy": float(np.std(accuracies))
}

metrics_dir = Path(f"results/metrics/cross_validation_{method}")
metrics_dir.mkdir(parents=True, exist_ok=True)

with open(metrics_dir / f"{method}_accuracy.json", "w") as f:
    json.dump(results, f, indent=4)

print("Accuracy per fold:", accuracies)
print("Mean accuracy:", np.mean(accuracies))
print("Standard deviation:", np.std(accuracies))

  0%|          | 0/5 [00:00<?, ?it/s]

Tokenizing texts...
Tokenization done! Time taken: 2.925 seconds
Dataloader done! Time taken: 0.390 seconds


  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.4351 | train acc: 0.7965 | val loss: 0.3805 | val acc: 0.8377
Epoch 02 | train loss: 0.4152 | train acc: 0.8119 | val loss: 0.3727 | val acc: 0.8393
Epoch 03 | train loss: 0.4109 | train acc: 0.8109 | val loss: 0.3667 | val acc: 0.8438
Epoch 04 | train loss: 0.4099 | train acc: 0.8122 | val loss: 0.3662 | val acc: 0.8421
Epoch 05 | train loss: 0.4030 | train acc: 0.8125 | val loss: 0.3563 | val acc: 0.8413
Epoch 06 | train loss: 0.4023 | train acc: 0.8115 | val loss: 0.3550 | val acc: 0.8419
Epoch 07 | train loss: 0.4033 | train acc: 0.8115 | val loss: 0.3560 | val acc: 0.8426
Epoch 08 | train loss: 0.3978 | train acc: 0.8144 | val loss: 0.3625 | val acc: 0.8408
Epoch 09 | train loss: 0.3994 | train acc: 0.8136 | val loss: 0.3577 | val acc: 0.8404
Epoch 10 | train loss: 0.3984 | train acc: 0.8151 | val loss: 0.3539 | val acc: 0.8416
Training done! Time taken: 18.383 seconds
Tokenizing texts...
Tokenization done! Time taken: 2.961 seconds
Dataloader done! Time t

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.4337 | train acc: 0.7973 | val loss: 0.3831 | val acc: 0.8308
Epoch 02 | train loss: 0.4142 | train acc: 0.8123 | val loss: 0.3711 | val acc: 0.8416
Epoch 03 | train loss: 0.4113 | train acc: 0.8118 | val loss: 0.3736 | val acc: 0.8423
Epoch 04 | train loss: 0.4101 | train acc: 0.8127 | val loss: 0.3548 | val acc: 0.8445
Epoch 05 | train loss: 0.4054 | train acc: 0.8146 | val loss: 0.3636 | val acc: 0.8353
Epoch 06 | train loss: 0.4035 | train acc: 0.8144 | val loss: 0.3531 | val acc: 0.8414
Epoch 07 | train loss: 0.4026 | train acc: 0.8121 | val loss: 0.3590 | val acc: 0.8425
Epoch 08 | train loss: 0.3992 | train acc: 0.8142 | val loss: 0.3607 | val acc: 0.8399
Epoch 09 | train loss: 0.4003 | train acc: 0.8125 | val loss: 0.3548 | val acc: 0.8434
Epoch 10 | train loss: 0.3979 | train acc: 0.8142 | val loss: 0.3500 | val acc: 0.8452
Training done! Time taken: 15.948 seconds
Tokenizing texts...
Tokenization done! Time taken: 2.943 seconds
Dataloader done! Time t

... (2 outputs left out)

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.4350 | train acc: 0.7965 | val loss: 0.3779 | val acc: 0.8370
Epoch 02 | train loss: 0.4144 | train acc: 0.8125 | val loss: 0.3695 | val acc: 0.8408
Epoch 03 | train loss: 0.4119 | train acc: 0.8125 | val loss: 0.3672 | val acc: 0.8373
Epoch 04 | train loss: 0.4076 | train acc: 0.8130 | val loss: 0.3747 | val acc: 0.8330
Epoch 05 | train loss: 0.4050 | train acc: 0.8125 | val loss: 0.3670 | val acc: 0.8361
Epoch 06 | train loss: 0.4020 | train acc: 0.8142 | val loss: 0.3614 | val acc: 0.8413
Epoch 07 | train loss: 0.4025 | train acc: 0.8124 | val loss: 0.3544 | val acc: 0.8397
Epoch 08 | train loss: 0.3992 | train acc: 0.8144 | val loss: 0.3556 | val acc: 0.8425
Epoch 09 | train loss: 0.3985 | train acc: 0.8158 | val loss: 0.3534 | val acc: 0.8402
Epoch 10 | train loss: 0.3991 | train acc: 0.8133 | val loss: 0.3518 | val acc: 0.8410
Training done! Time taken: 20.957 seconds
Tokenizing texts...
Tokenization done! Time taken: 3.215 seconds
Dataloader done! Time t

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.4340 | train acc: 0.7977 | val loss: 0.3884 | val acc: 0.8279
Epoch 02 | train loss: 0.4151 | train acc: 0.8133 | val loss: 0.3731 | val acc: 0.8426
Epoch 03 | train loss: 0.4113 | train acc: 0.8122 | val loss: 0.3680 | val acc: 0.8427
Epoch 04 | train loss: 0.4097 | train acc: 0.8108 | val loss: 0.3660 | val acc: 0.8348
Epoch 05 | train loss: 0.4034 | train acc: 0.8136 | val loss: 0.3605 | val acc: 0.8440
Epoch 06 | train loss: 0.4019 | train acc: 0.8134 | val loss: 0.3661 | val acc: 0.8353
Epoch 07 | train loss: 0.4025 | train acc: 0.8127 | val loss: 0.3593 | val acc: 0.8419
Epoch 08 | train loss: 0.4007 | train acc: 0.8134 | val loss: 0.3537 | val acc: 0.8464
Epoch 09 | train loss: 0.3974 | train acc: 0.8144 | val loss: 0.3558 | val acc: 0.8459
Epoch 10 | train loss: 0.4002 | train acc: 0.8143 | val loss: 0.3531 | val acc: 0.8441
Training done! Time taken: 19.165 seconds
Accuracy per fold: [0.8415738678544915, 0.8452115812917594, 0.8435783221974759, 0.840979

# What design choices have meaningful impact in classification performance (eg, embedding size, architecture design, optimisation parameters), if any?

> Mean-pooling performs way better than max-pooling.

>Adding more neurons does not improve the the classification performance

>Fasttext performs much better than Glove (embedding size 50 -> 300?): 

Fasttext: mean accuracy = 84.2%

Glove: mean accuracy = 75.39%

# Did pretrained embedding add to your design?

> No, the classification performance of mlp2 did not improve, it almost even and maybe worse for 1-2%

> The fact that increasing neurons increases the classification performance may suggest that the current model is too simple to capture semantic relationships provided by pretrained word embedding

# LSTM

In [ ]:
from part2.clslib import *
import part1.experiment as exp1
import part2.experiment as exp2

In [60]:
enc_type = "lstm"
tokenizer = BasicTokenizer(
    lowercase=True,
    remove_punct=False,
    remove_stopwords=False,
    vocab=trained_emb_model.vocab
)
enc_kwargs = {
    "hidden_size": 256,
    "freeze_embeddings": True,
    "word_dropout": 0.1
}

regressor_kwargs = {
    "hidden_size1": 128,
    "hidden_size2": 64,
    "dropout": 0.3
}

opt_kwargs = {
    "learning_rate": 0.001,
    "weight_decay": 1e-4,
    "batch_size": 64,
    "num_epochs": 10
}

**Baseline:** 

Encoder: hidden size = 128, frozen embeddings, word dropout = 0.1

Classifier: hidden sizes = 128, 64; dropout = 0.3

Optimizer: learning rate = 0.001, weight decay = 1e-4

Training: batch size = 64, epochs = 10

# Cross-validation

In [61]:
from pathlib import Path
import torch
import numpy as np
import json

Path("results/lstm/cross_validation/baseline").mkdir(parents=True, exist_ok=True)
train, validation, test = load_sst2()
data_splits = exp1.cross_validation_split(
    pairs=train,
    k=5,
    seed=1
)
method = 'baselinetest'

results_lstm = exp2.cross_validation(
    outcome_space=CLASSES,
    data_splits=data_splits,
    tokenizer=tokenizer,
    emb_model=trained_emb_model,
    enc_type="lstm",
    enc_kwargs=enc_kwargs,
    regressor_kwargs=regressor_kwargs,
    opt_kwargs=opt_kwargs,
    device="cuda" if torch.cuda.is_available() else "cpu",
    save_to=f"results/lstm/cross_validation/{method}"
)
accuracies = list(results_lstm["accuracies"].values())

results = {
    "accuracy_per_fold": accuracies,
    "mean_accuracy": float(np.mean(accuracies)),
    "std_accuracy": float(np.std(accuracies))
}

Path(f"results/metrics/cross_validation_{method}").mkdir(parents=True, exist_ok=True)

with open(f"results/metrics/cross_validation_{method}/hiddendims_accuracy.json", "w") as f:
    json.dump(results, f, indent=4)

accuracies = list(results_lstm["accuracies"].values())
print("Accuracy per fold:", accuracies)
print("Mean accuracy:", np.mean(accuracies))
print("Standard deviation:", np.std(accuracies))

  0%|          | 0/5 [00:00<?, ?it/s]

Tokenizing texts...
Tokenization done! Time taken: 3.092 seconds
Dataloader done! Time taken: 0.123 seconds


  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.4076 | train acc: 0.8159 | val loss: 0.3304 | val acc: 0.8610
Epoch 02 | train loss: 0.3386 | train acc: 0.8488 | val loss: 0.2904 | val acc: 0.8742
Epoch 03 | train loss: 0.3044 | train acc: 0.8689 | val loss: 0.2693 | val acc: 0.8891
Epoch 04 | train loss: 0.2741 | train acc: 0.8825 | val loss: 0.2428 | val acc: 0.9037
Epoch 05 | train loss: 0.2509 | train acc: 0.8963 | val loss: 0.2311 | val acc: 0.9113
Epoch 06 | train loss: 0.2341 | train acc: 0.9041 | val loss: 0.2143 | val acc: 0.9150
Epoch 07 | train loss: 0.2124 | train acc: 0.9131 | val loss: 0.2120 | val acc: 0.9189
Epoch 08 | train loss: 0.1982 | train acc: 0.9201 | val loss: 0.1982 | val acc: 0.9252
Epoch 09 | train loss: 0.1812 | train acc: 0.9281 | val loss: 0.1936 | val acc: 0.9276
Epoch 10 | train loss: 0.1709 | train acc: 0.9318 | val loss: 0.1908 | val acc: 0.9308
Training done! Time taken: 129.418 seconds
Tokenizing texts...
Tokenization done! Time taken: 3.360 seconds
Dataloader done! Time 

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.3999 | train acc: 0.8186 | val loss: 0.3229 | val acc: 0.8624
Epoch 02 | train loss: 0.3358 | train acc: 0.8532 | val loss: 0.2987 | val acc: 0.8752
Epoch 03 | train loss: 0.3050 | train acc: 0.8679 | val loss: 0.2550 | val acc: 0.8961
Epoch 04 | train loss: 0.2760 | train acc: 0.8839 | val loss: 0.2405 | val acc: 0.9045
Epoch 05 | train loss: 0.2539 | train acc: 0.8943 | val loss: 0.2228 | val acc: 0.9123
Epoch 06 | train loss: 0.2323 | train acc: 0.9049 | val loss: 0.2131 | val acc: 0.9166
Epoch 07 | train loss: 0.2126 | train acc: 0.9144 | val loss: 0.2082 | val acc: 0.9211
Epoch 08 | train loss: 0.1980 | train acc: 0.9200 | val loss: 0.1997 | val acc: 0.9278
Epoch 09 | train loss: 0.1823 | train acc: 0.9266 | val loss: 0.1966 | val acc: 0.9284
Epoch 10 | train loss: 0.1698 | train acc: 0.9337 | val loss: 0.1889 | val acc: 0.9301
Training done! Time taken: 120.063 seconds
Tokenizing texts...
Tokenization done! Time taken: 2.913 seconds
Dataloader done! Time 

... (2 outputs left out)

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.4076 | train acc: 0.8175 | val loss: 0.3250 | val acc: 0.8609
Epoch 02 | train loss: 0.3415 | train acc: 0.8483 | val loss: 0.2880 | val acc: 0.8807
Epoch 03 | train loss: 0.3074 | train acc: 0.8655 | val loss: 0.2737 | val acc: 0.8820
Epoch 04 | train loss: 0.2800 | train acc: 0.8793 | val loss: 0.2553 | val acc: 0.8943
Epoch 05 | train loss: 0.2536 | train acc: 0.8941 | val loss: 0.2339 | val acc: 0.9074
Epoch 06 | train loss: 0.2338 | train acc: 0.9027 | val loss: 0.2159 | val acc: 0.9189
Epoch 07 | train loss: 0.2134 | train acc: 0.9130 | val loss: 0.2149 | val acc: 0.9169
Epoch 08 | train loss: 0.1982 | train acc: 0.9207 | val loss: 0.2047 | val acc: 0.9258
Epoch 09 | train loss: 0.1842 | train acc: 0.9256 | val loss: 0.1932 | val acc: 0.9284
Epoch 10 | train loss: 0.1721 | train acc: 0.9312 | val loss: 0.2074 | val acc: 0.9285
Training done! Time taken: 172.072 seconds
Tokenizing texts...
Tokenization done! Time taken: 4.084 seconds
Dataloader done! Time 

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.4014 | train acc: 0.8194 | val loss: 0.3137 | val acc: 0.8646
Epoch 02 | train loss: 0.3417 | train acc: 0.8500 | val loss: 0.2833 | val acc: 0.8816
Epoch 03 | train loss: 0.3049 | train acc: 0.8661 | val loss: 0.2582 | val acc: 0.8938
Epoch 04 | train loss: 0.2725 | train acc: 0.8838 | val loss: 0.2446 | val acc: 0.9001
Epoch 05 | train loss: 0.2531 | train acc: 0.8951 | val loss: 0.2259 | val acc: 0.9104
Epoch 06 | train loss: 0.2306 | train acc: 0.9047 | val loss: 0.2173 | val acc: 0.9183
Epoch 07 | train loss: 0.2117 | train acc: 0.9140 | val loss: 0.2110 | val acc: 0.9200
Epoch 08 | train loss: 0.1935 | train acc: 0.9211 | val loss: 0.2071 | val acc: 0.9221
Epoch 09 | train loss: 0.1793 | train acc: 0.9267 | val loss: 0.2004 | val acc: 0.9289
Epoch 10 | train loss: 0.1693 | train acc: 0.9331 | val loss: 0.1993 | val acc: 0.9281
Training done! Time taken: 173.823 seconds
Accuracy per fold: [0.9308092056421677, 0.9301410541945063, 0.9246473645137342, 0.92843

# Hyper parameter search


1. What design choices have meaningful impact in classification performance, if any?

    **classifier hidden dims:**

        128 -> 256, 64 -> 128: less than 1% decrease in mean accuracy

        128 -> 512, 64 -> 256: still less than 1% decrease in mean accuracy
    
    **Encoder hidden dims**
    
        128 -> 256: less than 1% increase in mean accuracy
    
    **Word dropout**

        0.1-> 0.3: 1% decrease in mean accuracy

    **Classifier drop out**

        0.3 -> 0.1: No difference
    
    **Combined**

        No improvement in combinations

2. Did you investigate the impact of data size, embedding parameters (trained or not, fixed or trainable, etc)?
    * Freeze-embedding = False: No obvious difference in mean accuracy
3. Did you struggle with over-fitting?
    * Not in cross validaiton, but overfitting  found in whole train and validation set.
4. How far can this approach go? Are you able to improve meaningfully on embed-and-pool?
    * Yes there is a 8% increase in accuracy

# Validation model

In [ ]:
Path("results/lstm/validation").mkdir(parents=True, exist_ok=True)


train, validation, test = load_sst2()

metrics, best_epoch = exp2.train(
    training_set=train,
    validation_set=validation,
    tokenizer=tokenizer,
    emb_model=trained_emb_model,
    num_classes=len(CLASSES),
    enc_type="lstm",
    enc_kwargs=enc_kwargs,
    regressor_kwargs=regressor_kwargs,
    opt_kwargs=opt_kwargs,
    device="cuda" if torch.cuda.is_available() else "cpu",
    save_to="results/lstm/validation/baseline_model"
)

Path("results/metrics/validation").mkdir(parents=True, exist_ok=True)

accuracy = metrics[best_epoch]["val_metrics"]["accuracy"]

with open("results/metrics/validation/baseline_lstm_accuracy.json", "w") as f:
    json.dump({"accuracy": accuracy}, f)

print(f"Validation accuracy: {accuracy:.4f}")
print(f"Best epoch: {best_epoch + 1}")
print(f"Validation accuracy: {accuracy:.4f}")

Tokenizing texts...
Tokenization done! Time taken: 2.943 seconds
Dataloader done! Time taken: 0.317 seconds


  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.3973 | train acc: 0.8198 | val loss: 0.4419 | val acc: 0.8073
Epoch 02 | train loss: 0.3316 | train acc: 0.8532 | val loss: 0.3900 | val acc: 0.8257
Epoch 03 | train loss: 0.2998 | train acc: 0.8706 | val loss: 0.3588 | val acc: 0.8578
Epoch 04 | train loss: 0.2713 | train acc: 0.8843 | val loss: 0.3925 | val acc: 0.8463
Epoch 05 | train loss: 0.2486 | train acc: 0.8970 | val loss: 0.3450 | val acc: 0.8567
Epoch 06 | train loss: 0.2299 | train acc: 0.9063 | val loss: 0.3839 | val acc: 0.8567
Epoch 07 | train loss: 0.2131 | train acc: 0.9145 | val loss: 0.3681 | val acc: 0.8486
Epoch 08 | train loss: 0.2002 | train acc: 0.9205 | val loss: 0.3544 | val acc: 0.8578
Epoch 09 | train loss: 0.1909 | train acc: 0.9236 | val loss: 0.3749 | val acc: 0.8544
Epoch 10 | train loss: 0.1792 | train acc: 0.9294 | val loss: 0.3997 | val acc: 0.8567
Training done! Time taken: 96.818 seconds
Validation accuracy: 0.8567
Best epoch: 5
Validation accuracy: 0.8567



1. What design choices have meaningful impact in classification performance, if any?
    * Best model encoder hidden size = 256, word drop out = 0.1, freeze embedding true. Classifier hidden size 1 = 512, 2 = 256.  
    * Hidden size for classifier matters, 2% increase from 128->512 and 64-256
    * Encoder hidden size matters, 1% increase from 128->256
    * Encoder word drop out 0.1->0.3: 2% percent decrease
    * Classifier drop out 0.3->0.1: around 2% decrease
2. Did you investigate the impact of data size, embedding parameters (trained or not, fixed or trainable, etc)?
    * Freeze-embedding = False: 2-3% decrease in accuracy
3. Did you struggle with over-fitting?
    * There is a difference of 5-6% in accuracy between train and validation, which suggest there is an overfitting, but no hard evidence.
4. How far can this approach go? Are you able to improve meaningfully on embed-and-pool?
    * Yes there is a 5% increase in accuracy

# 5. We make decisions under uncertainty all the time, are you gathering enough information to substantiate your choices? Various things can lead to small differences in results, and these differences are not always consistent. Here are some things to think about:
    
**How did you structure you hyperparameter search?**

Hyperparameters are changed one at the time and compared to the baseline, also combinations of hyperparameters are applied to see if any improvements.

**Are you keeping an eye on experimental variance?**

yes, different seed were tested and the results are not changed a lot

# 6. Experiment with error analysis and try to discover what characteristics of input texts are predictive of performance differences across major design choices. 

**What analysis features did you come up with?**

1. sentence length
2. Negation
3. Contrast
4. rare words

**Which ones did you manage to test and how did that go?**

1. sentence length
2. Negation
3. Contrast
4. rare words

**What would you prioritise in the future in order to improve your systems?**

We would prioritise improving performance on sentences containing negation and contrast, as both models struggled with these features.

In [ ]:
import re
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

from part2 import experiment as exp2, analysis
from part2.clslib import BasicTokenizer

train, validation, _ = load_sst2()
texts, labels = zip(*validation)
labels = np.asarray(labels, dtype=int)

results = Path("results")
systems = {
    "embed-and-pool": (
        results / "embed_and_pool/cross_validation/fasttext_embed_and_pool/fold-0/model",
        trained_emb_model,
    ),
    "lstm": (
        results / "lstm/cross_validation/baselinetest/fold-0/model",
        trained_emb_model,
    ),
}

# Feature extraction
word_pattern = re.compile(r"[a-z]+(?:'[a-z]+)*")
def words(text):
    return word_pattern.findall(text.lower().replace("’", "'"))

word_counts = Counter(w for text, _ in train for w in words(text))
negation_words = {"no", "not", "never", "neither", "nor"}
contrast_words = {"but", "although", "though", "however", "yet"}

feature_names = ["token_count", "has_negation", "has_contrast", "rare_word_ratio"]
feature_rows = []

for text in texts:
    tokens = words(text)
    feature_rows.append({
        "token_count": len(tokens),
        "has_negation": int(any(w in negation_words or w.endswith("n't") for w in tokens)),
        "has_contrast": int(any(w in contrast_words for w in tokens)),
        "rare_word_ratio": sum(word_counts[w] <= 5 for w in tokens) / max(len(tokens), 1),
    })

items = pd.DataFrame(feature_rows).assign(
    item_id=np.arange(len(texts)), text=list(texts), label=labels
)

items["length_group"] = pd.cut(
    items["token_count"], [0, 10, 20, np.inf],
    labels=["short", "medium", "long"]
)
items["rare_group"] = pd.cut(
    items["rare_word_ratio"], [-0.001, 0.1, 0.3, 1],
    labels=["low", "medium", "high"]
)

predictions = []
for system_id, (name, (checkpoint, embeddings)) in enumerate(systems.items()):
    tokenizer = BasicTokenizer(
        lowercase=True, remove_punct=False,
        remove_stopwords=False, vocab=embeddings.vocab
    )
    model = exp2.load(checkpoint, embeddings, device="cpu")
    pred = exp2.predict_logprobs(
        model, tokenizer, texts, batch_size=64, device="cpu"
    ).argmax(axis=-1)

    predictions.append(items.assign(
        system=name, system_id=system_id,
        pred=pred, correct=(pred == labels).astype(int)
    ))

df = pd.concat(predictions, ignore_index=True)

for feature in ["length_group", "has_negation", "has_contrast", "rare_group"]:
    summary = df.groupby([feature, "system"], observed=True)["correct"].agg(
        n="size", accuracy="mean"
    ).unstack("system")

    acc = summary["accuracy"]
    print(f"\n{feature}")
    display(pd.DataFrame({
        "n_zinnen": summary["n"]["lstm"],
        "pooling_pct": 100 * acc["embed-and-pool"],
        "lstm_pct": 100 * acc["lstm"],
        "verschil_pp": 100 * (acc["lstm"] - acc["embed-and-pool"])
    }).round(2))

# Bayesian Data Analysis



length_group


,n_zinnen,pooling_pct,lstm_pct,verschil_pp
length_group,,,,
short,191,87.43,90.58,3.14
medium,375,78.40,85.87,7.47
long,306,75.82,82.68,6.86



has_negation


,n_zinnen,pooling_pct,lstm_pct,verschil_pp
has_negation,,,,
0,704,82.53,88.21,5.68
1,168,66.67,75.60,8.93



has_contrast


,n_zinnen,pooling_pct,lstm_pct,verschil_pp
has_contrast,,,,
0,742,80.86,86.52,5.66
1,130,71.54,81.54,10.00



rare_group


,n_zinnen,pooling_pct,lstm_pct,verschil_pp
rare_group,,,,
low,595,81.18,85.38,4.20
medium,267,76.03,86.52,10.49
high,10,70.00,90.00,20.00


In [16]:
features = df[feature_names].copy()
for col in ["token_count", "rare_word_ratio"]:
    std = features[col].std()
    if std > 0:
        features[col] = (features[col] - features[col].mean()) / std

mcmc = analysis.fit_bda_model(
    num_systems=len(systems),
    sys_id=df["system_id"].to_numpy(),
    features=features.to_numpy(),
    correctness=df["correct"].to_numpy(),
    num_chains=2,
)

weights = np.asarray(mcmc.get_samples()["w"])
difference = weights[:, 1, :] - weights[:, 0, :]
lower, median, upper = np.quantile(difference, [0.025, 0.5, 0.975], axis=0)

display(pd.DataFrame({
    "feature": feature_names,
    "median_lstm_minus_pooling": median,
    "lower_95": lower,
    "upper_95": upper,
}).round(3))

sample: 100%|██████████| 2000/2000 [00:02<00:00, 852.83it/s, 31 steps of size 6.99e-02. acc. prob=0.94] 


,feature,median_lstm_minus_pooling,lower_95,upper_95
0,token_count,-0.003,-0.282,0.254
1,has_negation,0.005,-0.220,0.249
2,has_contrast,0.065,-0.168,0.313
3,rare_word_ratio,0.175,-0.064,0.420
